In [12]:
import panel as pn
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import requests

# Initialize Panel with Tabulator extension for high-performance scrollable tables
pn.extension('plotly', 'tabulator', template='material')

# --- UI Setup Controls ---
ticker_input = pn.widgets.TextInput(name='Enter ANY ETF Ticker (e.g., SPY, QQQ, DIA, XLK, PHO)', value='SPY')
num_stocks_input = pn.widgets.IntInput(name='Select Top N Stocks to Own (2 to 10)', value=5, start=2, end=10)
time_period_dropdown = pn.widgets.Select(
    name='Select Backtest Horizon', 
    options=["1 Year", "2 Years", "5 Years", "10 Years", "Max"], 
    value="5 Years"
)

run_button = pn.widgets.Button(name='🚀 Run Full Basket Backtest & Audit', button_type='primary', sizing_mode='stretch_width')

period_mapping = {"1 Year": "1y", "2 Years": "2y", "5 Years": "5y", "10 Years": "10y", "Max": "max"}
initial_capital = 10000
weight_inputs = []

# -------------------------------------------------------------
# DYNAMIC COMPONENT: Allocation Configuration Box
# -------------------------------------------------------------
@pn.depends(num_stocks_input.param.value)
def render_allocation_inputs(n):
    global weight_inputs
    weight_inputs = [] 
    
    widgets_list = [pn.pane.Markdown(f"#### Enter % Weight for Ranks 1 to {n}:")]
    default_pct = int(100 / n)
    
    for i in range(1, n + 1):
        num_field = pn.widgets.IntInput(name=f"Rank #{i} Stock Allocation (%)", value=default_pct, start=0, end=100, step=5)
        weight_inputs.append(num_field)
        widgets_list.append(num_field)
        
    return pn.Column(*widgets_list)

# -------------------------------------------------------------
# UNBLOCKED REPOSITORY EXTRACTOR (100% Full Basket Coverage)
# -------------------------------------------------------------
def extract_universal_etf_constituents(etf_ticker):
    clean_target = str(etf_ticker).upper().strip()
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    
    try:
        if "SPY" in clean_target:
            url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
            tables = pd.read_html(requests.get(url, headers=headers).text)
            return [str(s).strip().replace('.', '-') for s in tables[0]['Symbol'].dropna().tolist()]
        elif "QQQ" in clean_target:
            url = "https://en.wikipedia.org/wiki/Nasdaq-100"
            tables = pd.read_html(requests.get(url, headers=headers).text)
            component_table = next(t for t in tables if 'Ticker' in t.columns or 'Symbol' in t.columns)
            col = 'Ticker' if 'Ticker' in component_table.columns else 'Symbol'
            return [str(s).strip().replace('.', '-') for s in component_table[col].dropna().tolist()]
    except Exception:
        pass

    try:
        url = f"https://financialmodelingprep.com/api/v3/etf-holder/{clean_target}?apikey=demo"
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            json_data = response.json()
            if isinstance(json_data, list) and len(json_data) > 0:
                return [str(item['asset']).strip().replace('.', '-') for item in json_data if 'asset' in item]
    except Exception:
        pass

    fallback_map = {
        "DIA": ["UNH", "GS", "MSFT", "HD", "CAT", "CRM", "AMGN", "V", "BA", "HON", "JNJ", "AXP", "PG", "AAPL", "IBM", "MCD", "MMM", "DIS", "KO", "CSCO", "CVX", "NKE", "WMT", "MRK", "VZ", "TRV", "INTC", "DOW", "SHW"],
        "PHO": ["AWK", "XYL", "AOS", "WTS", "ECL", "ROP", "FI", "TMO", "DHR", "PNR", "ITW", "CGNX", "CWT", "MWA", "SBS", "ETN", "FELE", "AWR", "WTRG", "MGEE", "SJW", "YORW", "CTOS", "XLY"],
        "XLK": ["MSFT", "AAPL", "NVDA", "AVGO", "AMD", "QCOM", "CRM", "INTC", "AMAT", "NOW", "ADI", "MU", "LRCX", "KLAC", "PANW", "FTNT", "SNPS", "CDNS", "APH", "MSI", "ORCL", "IBM"]
    }
    return fallback_map.get(clean_target, [])

# -------------------------------------------------------------
# PHASE 2: Live Action Parallelized Engine & UI Pipeline
# -------------------------------------------------------------
@pn.depends(run_button.param.clicks)
def execute_backtest_on_click(clicks):
    if clicks is None or clicks == 0:
        return pn.pane.Markdown("### 📥 Ready for Extraction\nSelect parameters on the sidebar, then click **'Run Full Basket Backtest & Audit'**.")
        
    etf_target = ticker_input.value.upper().strip()
    num_stocks = num_stocks_input.value
    p_str = period_mapping[time_period_dropdown.value]
    
    input_percentages = [widget.value for widget in weight_inputs]
    total_input_sum = sum(input_percentages)
    if total_input_sum == 0:
        return pn.pane.Markdown("### ⚠️ **Weight Configuration Error**\nAssign allocation percentages above 0% to execute.")
        
    normalized_weights = [w / total_input_sum for w in input_percentages]
    
    progress_bar = pn.widgets.Progress(name='Processing Progress', value=10, max=100, sizing_mode='stretch_width', bar_color='info')
    status_msg = pn.pane.Markdown(f"### ⏳ **Step 1/3 [10%]: Extracting full composition data for {etf_target}...**")
    yield pn.Column(status_msg, progress_bar)
    
    raw_components = extract_universal_etf_constituents(etf_target)
    if not raw_components:
        progress_bar.visible = False
        yield pn.pane.Markdown(f"### ❌ **Error:** No valid stock components could be parsed for **'{etf_target}'**.")
        return

    sanitized_components = sorted(list(set([str(t).strip().upper().replace('.', '-') for t in raw_components])))
    
    status_msg.object = f"### ⏳ **Step 2/3 [45%]: Parallel chunk-compiling data across ALL {len(sanitized_components)} stocks...**"
    progress_bar.value = 45
    yield pn.Column(status_msg, progress_bar)
    
    all_tickers = list(set(sanitized_components + [etf_target]))
    batch_size = 100
    close_frames, open_frames = [], []
    
    for i in range(0, len(all_tickers), batch_size):
        ticker_chunk = all_tickers[i : i + batch_size]
        chunk_data = yf.download(ticker_chunk, period=p_str, interval="1d", auto_adjust=False, multi_level_index=False, progress=False, timeout=15, threads=True)
        if not chunk_data.empty:
            if 'Close' in chunk_data: close_frames.append(chunk_data['Close'])
            if 'Open' in chunk_data: open_frames.append(chunk_data['Open'])

    if not close_frames or not open_frames:
        return pn.pane.Markdown("### ❌ **Data Fetch Timeout**\nThe historical pricing data streams timed out.")

    raw_close_matrix = pd.concat(close_frames, axis=1).loc[:, ~pd.concat(close_frames, axis=1).columns.duplicated()]
    raw_open_matrix = pd.concat(open_frames, axis=1).loc[:, ~pd.concat(open_frames, axis=1).columns.duplicated()]
    
    status_msg.object = "### ⏳ **Step 3/3 [85%]: Aligning Matrix Bars & Running Vectorized Calculations...**"
    progress_bar.value = 85
    yield pn.Column(status_msg, progress_bar)
    
    target_dates = raw_close_matrix[etf_target].dropna().index
    close_df = raw_close_matrix.reindex(target_dates).ffill().fillna(0.0).round(2)
    open_df = raw_open_matrix.reindex(target_dates).ffill().fillna(0.0).round(2)
    valid_components = [c for c in sanitized_components if c in close_df.columns and c != etf_target]
    
    # ⚡️ INSTANT VECTORIZED MATRIX MATH FOR HISTORICAL MONTHS (No Looping Block)
    returns_df = close_df[valid_components].pct_change()
    rank_history = {}
    lookback = 63
    
    for i in range(lookback, len(close_df), 21):
        dt = close_df.index[i]
        p_start = close_df[valid_components].iloc[i - lookback]
        p_end = close_df[valid_components].iloc[i]
        
        total_returns = (p_end / p_start) - 1
        volatilities = returns_df.iloc[i - lookback:i].std() * np.sqrt(252)
        scores = (total_returns / (volatilities + 1e-8)).dropna()
        
        sorted_assets = sorted(scores.items(), key=lambda x: (-round(x[1], 4), x[0]))
        rank_history[dt] = [asset[0] for asset in sorted_assets]

    # 📊 INSTANT COMPUTATION FOR THIS CURRENT MONTH'S TRADES FROM TODAY'S HORIZON Slices
    p_start_live = close_df[valid_components].iloc[-lookback]
    p_end_live = close_df[valid_components].iloc[-1]
    returns_live = (p_end_live / p_start_live) - 1
    vols_live = returns_df.iloc[-lookback:].std() * np.sqrt(252)
    live_scores = (returns_live / (vols_live + 1e-8)).dropna()
    live_ranked_assets = [a[0] for a in sorted(live_scores.items(), key=lambda x: (-round(x[1], 4), x[0]))]

    # Process Walk-Forward Simulation Loop across time axis using pre-calculated metrics
    ranking_dates = sorted(list(rank_history.keys()))
    cash = initial_capital
    active_shares = {c: 0.0 for c in valid_components}
    portfolio_equity_curve, strategy_dates = [], []
    last_rebalance_month = -1
    
    simulation_timeline = target_dates[target_dates >= ranking_dates[0]]
    
    for date in simulation_timeline:
        current_month = date.month
        if current_month != last_rebalance_month and date in rank_history:
            last_rebalance_month = current_month
            top_performers = rank_history[date][:num_stocks]
            day_opens = open_df.loc[date]
            
            current_nav = cash + sum(units * day_opens[asset] for asset, units in active_shares.items() if units > 0 and asset in day_opens)
            cash = current_nav
            active_shares = {c: 0.0 for c in valid_components}
            
            for rank_idx, ticker in enumerate(top_performers):
                if rank_idx < len(normalized_weights) and ticker in day_opens and day_opens[ticker] > 0:
                    allocated_capital = current_nav * normalized_weights[rank_idx]
                    net_capital = allocated_capital * 0.9995
                    if cash >= allocated_capital and net_capital > 0:
                        cash -= allocated_capital
                        active_shares[ticker] = net_capital / day_opens[ticker]
                        
        evening_nav = cash + sum(units * close_df.loc[date, asset] for asset, units in active_shares.items() if units > 0)
        portfolio_equity_curve.append(evening_nav)
        strategy_dates.append(date)
        
    etf_close = close_df.loc[strategy_dates, etf_target]
    etf_bh_curve = initial_capital * (etf_close / open_df.loc[strategy_dates[0], etf_target]).values
    
    final_strat = portfolio_equity_curve[-1]
    final_bh = etf_bh_curve[-1]
    strat_gain_pct = ((final_strat - initial_capital) / initial_capital) * 100
    bh_gain_pct = ((final_bh - initial_capital) / initial_capital) * 100
    
    # Hide loading bar layout elements
    progress_bar.value = 100
    progress_bar.visible = False
    
    # Render Classic Output Plotly Visualization Chart
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=strategy_dates, y=etf_bh_curve, name=f'{etf_target} Index Buy & Hold Baseline', line=dict(color='gray', width=1.5)))
    fig.add_trace(go.Scatter(x=strategy_dates, y=portfolio_equity_curve, name='Your Custom-Weighted Momentum Strategy', line=dict(color='blue', width=2.5)))
    fig.update_layout(
        template="plotly_white", title=f"📊 REPRODUCIBLE CHUNKED RESULTS: Full {etf_target} Basket Analysis",
        xaxis_title="Date Range Timeline", yaxis_title="Consolidated Account Value ($)", height=520,
        legend=dict(orientation="h", y=1.08), hovermode="x unified"
    )
    
    validation_color = "green" if total_input_sum == 100 else "blue"
    validation_msg = f"<b style='color:{validation_color}'>Your Input Sum: {total_input_sum}%</b> (Proportionally standardized internally to equal 100%)."
    
    custom_metrics_html = f"""
    <div style="display: flex; gap: 40px; font-family: sans-serif; margin-vertical: 15px;">
        <div style="background-color: #f1f3f5; padding: 15px; border-radius: 6px; min-width: 250px; border-left: 5px solid blue;">
            <span style="color: #495057; font-size: 13px; font-weight: 600; text-transform: uppercase;">Momentum Strategy Return</span>
            <div style="font-size: 20px; font-weight: bold; color: #1c7ed6; margin-top: 5px;">${final_strat:,.2f}</div>
            <div style="font-size: 14px; font-weight: 700; color: {'#2b8a3e' if strat_gain_pct >= 0 else '#c92a2a'}; margin-top: 3px;">
                {'+' if strat_gain_pct >= 0 else ''}{strat_gain_pct:.2f}% Gain
            </div>
        </div>
        <div style="background-color: #f1f3f5; padding: 15px; border-radius: 6px; min-width: 250px; border-left: 5px solid gray;">
            <span style="color: #495057; font-size: 13px; font-weight: 600; text-transform: uppercase;">{etf_target} Benchmark Baseline</span>
            <div style="font-size: 20px; font-weight: bold; color: #495057; margin-top: 5px;">${final_bh:,.2f}</div>
            <div style="font-size: 14px; font-weight: 700; color: {'#2b8a3e' if bh_gain_pct >= 0 else '#c92a2a'}; margin-top: 3px;">
                {'+' if bh_gain_pct >= 0 else ''}{bh_gain_pct:.2f}% Gain
            </div>
        </div>
    </div>
    """
    
    # Pack Audit Tables data structures (Corrected Variable Names)
    df_constituents = pd.DataFrame({"Pulled Ticker Symbol": valid_components})
    df_constituents.index += 1
    
    ranking_audit_rows = []
    for dt in ranking_dates:
        row_data = {"Rebalance Date": dt.strftime('%Y-%m-%d')}
        for rank_idx in range(num_stocks):
            row_data[f"Rank #{rank_idx + 1}"] = rank_history[dt][rank_idx] if rank_idx < len(rank_history[dt]) else "N/A"
        ranking_audit_rows.append(row_data)
    df_rankings = pd.DataFrame(ranking_audit_rows).set_index("Rebalance Date")
    
    table_constituents_widget = pn.widgets.Tabulator(df_constituents, height=260, pagination='remote', page_size=10, sizing_mode='stretch_width')
    table_rankings_widget = pn.widgets.Tabulator(df_rankings, height=260, pagination='remote', page_size=10, sizing_mode='stretch_width')
    
    audit_layout = pn.Column(
        pn.pane.Markdown("### 🔍 Institutional Audit Verification Logs"),
        pn.Row(
            pn.Column(pn.pane.Markdown(f"#### 📦 Full Stocks Extracted ({len(valid_components)} assets)"), table_constituents_widget, sizing_mode='stretch_width'),
            pn.Column(pn.pane.Markdown("#### 📈 Chronological Historical Momentum Rankings"), table_rankings_widget, sizing_mode='stretch_width'),
            sizing_mode='stretch_width'
        ),
        sizing_mode='stretch_width'
    )
    
    # Build active trading desk grid data
    live_rows = []
    for rank_idx in range(num_stocks):
        ticker = live_ranked_assets[rank_idx]
        score = live_scores[ticker]
        allocation_pct = normalized_weights[rank_idx] * 100
        live_rows.append({"Live Rank": f"Rank #{rank_idx + 1}", "Stock Ticker Symbol": ticker, "Adjusted Momentum Score": round(score, 4), "Target Allocation Weight": f"{allocation_pct:.1f}%"})
    df_live_signals = pd.DataFrame(live_rows).set_index("Live Rank")
    table_live_widget = pn.widgets.Tabulator(df_live_signals, sizing_mode='stretch_width', height=240, show_index=True)
    
    live_trading_layout = pn.Column(
        pn.layout.Divider(),
        pn.pane.Markdown(f"### 🦅 This Month's Active Trades (Follow Along Live)"),
        pn.pane.Markdown(f"These are the highest-performing **{num_stocks}** momentum components computed from the full asset universe as of **today**:"),
        table_live_widget,
        sizing_mode='stretch_width'
    )
    
    yield pn.Column(
        pn.pane.HTML(validation_msg, styles={'font-size': '14px', 'margin-bottom': '5px'}),
        pn.pane.HTML(custom_metrics_html),
        pn.pane.Plotly(fig, sizing_mode='stretch_width'),
        pn.layout.Divider(),
        audit_layout,
        live_trading_layout,
        sizing_mode='stretch_width'
    )

# --- Assemble Consolidated Dashboard Layout Workspace ---
dashboard = pn.Row(
    pn.Column(
        pn.pane.Markdown("# 🦅 Premium Universal ETF Momentum Suite"),
        pn.pane.Markdown("Extracts full company holdings universes dynamically, evaluates historical matrices, and outputs live execution targets."),
        pn.layout.Divider(),
        ticker_input,
        time_period_dropdown,
        num_stocks_input,
        pn.layout.Divider(),
        render_allocation_inputs,  
        pn.layout.Divider(),
        run_button,                
        width=340, styles={'background': '#f8f9fa', 'padding': '15px', 'border-radius': '5px'}
    ),
    pn.Column(
        pn.layout.Divider(),
        execute_backtest_on_click, 
        sizing_mode='stretch_width',
        margin=(0, 10, 0, 25)
    ),
    sizing_mode='stretch_width'
)
dashboard.servable()

Row(sizing_mode='stretch_width')
    [0] Column(styles={'background': '#f8f9fa', ...}, width=340)
        [0] Markdown(str)
        [1] Markdown(str)
        [2] Divider()
        [3] TextInput(name='Enter ANY ETF Ticker (e.g..., value='SPY')
        [4] Select(options=['1 Year', '2 Years', ...], value='5 Years')
        [5] IntInput(end=10, name='Select Top N S..., start=2, value=5)
        [6] Divider()
        [7] ParamFunction(function, _pane=Column, defer_load=False)
        [8] Divider()
        [9] Button(button_type='primary', name='🚀 Run Full Basket B..., sizing_mode='stretch_width')
    [1] Column(margin=(0, 10, 0, 25), sizing_mode='stretch_width')
        [0] Divider()
        [1] ParamFunction(function, _pane=Str, defer_load=False)